# Week 2 Exercise: Local AI Customer Support Triage Assistant

This notebook completes the Week 2 end-of-week exercise with a business-focused prototype: an AI assistant for e-commerce customer support triage.

The prototype includes:

- A Gradio UI
- Streaming responses
- A system prompt that gives the assistant support operations expertise
- Model switching between local Ollama models
- Simple tools for ticket classification and mock order lookup
- Optional local microphone/upload audio input and local audio output

Before running it, make sure Ollama is running locally and at least one model is available:

```bash
ollama serve
ollama pull qwen2.5
ollama pull mistral
ollama pull llama3.2
# Optional local speech-to-text:
uv add faster-whisper
```

## Business Idea

Customer support teams lose time reading repetitive tickets, identifying urgency, checking basic order details, and drafting replies. This assistant acts as a first-pass triage layer. It gives the support agent a structured summary, recommended priority, likely category, next action, and a ready-to-send customer response.

This is commercially useful because it can reduce first-response time, standardize service quality, and help teams handle higher ticket volume without immediately hiring more agents.

In [ ]:
from IPython.display import Markdown, display
from openai import OpenAI
import gradio as gr
import json
import requests
import shutil
import subprocess
import tempfile
import uuid

OLLAMA_BASE_URL = "http://localhost:11434/v1"
OLLAMA_HEALTH_URL = "http://localhost:11434/"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

## Optional Local Audio

Ollama does not process raw audio itself, so this notebook uses a local speech-to-text layer before sending text to Ollama. The code supports `faster-whisper` first, then `openai-whisper` as a fallback. For speech output, it uses macOS `say`, then converts the generated audio to WAV with `afconvert` so Gradio/browser playback works more reliably.

The UI supports microphone and upload audio. If the embedded notebook view cannot detect your microphone, use the external browser tab opened by Gradio and allow microphone permission there.

If audio input says Whisper is missing, install one local STT package in your course environment, then rerun the imports and UI cells:

```bash
uv add faster-whisper
```

## Check Ollama

Run this cell first. If it fails, start Ollama in a terminal with `ollama serve`.

In [ ]:
def check_ollama():
    try:
        response = requests.get(OLLAMA_HEALTH_URL, timeout=3)
        response.raise_for_status()
        return "Ollama is running."
    except requests.RequestException as exc:
        raise RuntimeError(
            "Ollama does not seem to be running. Start it with `ollama serve`, "
            "then pull at least one model such as `ollama pull qwen2.5`."
        ) from exc

display(Markdown(check_ollama()))

## Local Models

These are the local Ollama models this prototype can switch between. Keep only the models you have pulled locally.

In [ ]:
MODEL_OPTIONS = [
    "qwen2.5:latest",
    "mistral:latest",
    "llama3.2:latest",
    "phi4-mini:latest",
    "gpt-oss:20b",
]

DEFAULT_MODEL = "qwen2.5:latest"

## Mock Business Data and Tools

In a real company these tools would call order-management, CRM, inventory, or helpdesk APIs. For the prototype, small local dictionaries are enough to demonstrate the pattern.

In [ ]:
ORDERS = {
    "A1001": {
        "customer": "Maya Chen",
        "item": "AeroFit Wireless Earbuds",
        "status": "Delivered",
        "delivery_date": "2026-05-02",
        "price": "$89",
        "warranty": "12 months",
        "notes": "Customer bought express shipping and protection plan.",
    },
    "A1002": {
        "customer": "Jon Bell",
        "item": "VoltPro Laptop Charger USB-C 100W",
        "status": "In transit",
        "delivery_date": "Expected 2026-05-08",
        "price": "$49",
        "warranty": "6 months",
        "notes": "Carrier scan delayed by 24 hours.",
    },
    "A1003": {
        "customer": "Sara Okafor",
        "item": "HomeBrew Espresso Starter Kit",
        "status": "Delivered",
        "delivery_date": "2026-04-29",
        "price": "$179",
        "warranty": "24 months",
        "notes": "Replacement parts available in stock.",
    },
    "A1004": {
        "customer": "Liam Rodriguez",
        "item": "DeskMate Standing Desk Converter",
        "status": "Return requested",
        "delivery_date": "2026-04-25",
        "price": "$129",
        "warranty": "12 months",
        "notes": "Return label generated but not scanned yet.",
    },
}

POLICIES = {
    "returns": "Returns are accepted within 30 days if the item is unused or defective. Defective items qualify for free return shipping.",
    "replacement": "Replacement can be offered for damaged, defective, or missing items when stock is available.",
    "shipping": "Express shipping delays over 48 hours qualify for a shipping-fee refund.",
    "refund": "Refunds are processed 3-5 business days after the returned item is scanned by the carrier.",
}


def lookup_order(order_id):
    order_id = (order_id or "").strip().upper()
    if not order_id:
        return {"found": False, "message": "No order ID provided."}
    order = ORDERS.get(order_id)
    if not order:
        return {"found": False, "message": f"No order found for {order_id}."}
    return {"found": True, "order_id": order_id, **order}


def classify_ticket(message):
    text = (message or "").lower()

    categories = []
    if any(word in text for word in ["broken", "damaged", "defective", "not working", "faulty"]):
        categories.append("damaged_or_defective_item")
    if any(word in text for word in ["late", "delay", "where is", "tracking", "not arrived", "delivery"]):
        categories.append("shipping_or_delivery")
    if any(word in text for word in ["refund", "return", "money back", "cancel"]):
        categories.append("refund_or_return")
    if any(word in text for word in ["invoice", "charged", "billing", "payment", "card"]):
        categories.append("billing")
    if any(word in text for word in ["how do i", "setup", "manual", "compatible", "recommend"]):
        categories.append("product_question")

    if not categories:
        categories.append("general_support")

    urgent_words = ["urgent", "today", "asap", "before friday", "tomorrow", "angry", "unacceptable"]
    urgency = "high" if any(word in text for word in urgent_words) else "normal"

    negative_words = ["angry", "frustrated", "terrible", "unacceptable", "disappointed", "annoyed"]
    sentiment = "negative" if any(word in text for word in negative_words) else "neutral"

    return {
        "categories": categories,
        "urgency": urgency,
        "sentiment": sentiment,
    }


def relevant_policies(classification):
    categories = classification["categories"]
    selected = {}
    if "damaged_or_defective_item" in categories:
        selected["replacement"] = POLICIES["replacement"]
        selected["returns"] = POLICIES["returns"]
    if "shipping_or_delivery" in categories:
        selected["shipping"] = POLICIES["shipping"]
    if "refund_or_return" in categories:
        selected["refund"] = POLICIES["refund"]
        selected["returns"] = POLICIES["returns"]
    return selected or POLICIES

## Expert System Prompt

The system prompt gives the model a clear role, expected output structure, and support-quality rules.

In [ ]:
SYSTEM_PROMPT = """
You are a senior customer support operations assistant for an e-commerce company.
Your job is to help a human support agent triage customer tickets quickly and responsibly.

Use the provided tool context. Do not invent order facts that are not in the context.
If the order data is missing, say what information the agent should request next.

Return a concise, practical response with these sections:
1. Triage summary
2. Priority and category
3. Recommended next action
4. Draft customer reply
5. Internal note for the support agent

Tone rules:
- Be calm, precise, and empathetic.
- Do not overpromise refunds, replacements, or delivery dates.
- Prefer clear operational next steps over vague reassurance.
""".strip()

## Streaming Assistant Logic

The tools run first, then their output is passed to the local model. The model streams the final support recommendation back into Gradio.

In [ ]:
def build_user_prompt(customer_message, order_id):
    classification = classify_ticket(customer_message)
    order_context = lookup_order(order_id)
    policy_context = relevant_policies(classification)

    tool_context = {
        "ticket_classification": classification,
        "order_lookup": order_context,
        "relevant_policies": policy_context,
    }

    prompt = f"""
Customer message:
{customer_message}

Order ID provided by agent:
{order_id or "No order ID provided"}

Tool context:
{json.dumps(tool_context, indent=2)}

Create the triage recommendation now.
""".strip()

    return prompt, tool_context


def stream_triage(customer_message, order_id, model):
    if not customer_message or not customer_message.strip():
        yield "Please enter a customer message."
        return

    prompt, tool_context = build_user_prompt(customer_message, order_id)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]

    try:
        stream = ollama.chat.completions.create(
            model=model,
            messages=messages,
            temperature=0.3,
            stream=True,
        )
    except Exception as exc:
        yield (
            f"Could not reach Ollama or model `{model}`. "
            "Make sure Ollama is running and the selected model is pulled locally.\n\n"
            f"Error: {exc}"
        )
        return

    response = ""
    tool_summary = (
        "### Tool context used\n"
        f"```json\n{json.dumps(tool_context, indent=2)}\n```\n\n"
        "### Assistant recommendation\n"
    )

    yield tool_summary

    for chunk in stream:
        delta = chunk.choices[0].delta.content or ""
        if not delta:
            continue
        response += delta
        yield tool_summary + response


def transcribe_audio_locally(audio_path):
    if not audio_path:
        return ""

    try:
        from faster_whisper import WhisperModel

        model = WhisperModel("base", device="cpu", compute_type="int8")
        segments, _ = model.transcribe(audio_path, beam_size=5)
        return " ".join(segment.text.strip() for segment in segments).strip()
    except ModuleNotFoundError:
        pass

    try:
        import whisper

        model = whisper.load_model("base")
        result = model.transcribe(audio_path)
        return result.get("text", "").strip()
    except ModuleNotFoundError as exc:
        raise RuntimeError(
            "Local speech-to-text is not installed. Install it with `uv add faster-whisper`, "
            "then rerun this notebook. Ollama handles the text reasoning, while Whisper handles audio transcription."
        ) from exc


def text_to_speech_locally(text):
    if not text or not text.strip():
        return None
    if not shutil.which("say"):
        return None

    base_path = f"{tempfile.gettempdir()}/support_triage_{uuid.uuid4().hex}"
    aiff_path = f"{base_path}.aiff"
    wav_path = f"{base_path}.wav"

    subprocess.run(["say", "-o", aiff_path, text[:3500]], check=True)

    if shutil.which("afconvert"):
        subprocess.run(
            ["afconvert", "-f", "WAVE", "-d", "LEI16", aiff_path, wav_path],
            check=True,
        )
        return wav_path

    return aiff_path


def stream_triage_with_audio(audio_path, typed_message, order_id, model, speak_response):
    try:
        transcript = transcribe_audio_locally(audio_path) if audio_path else ""
    except Exception as exc:
        yield f"Audio transcription failed: {exc}", "", None
        return

    customer_message = transcript or typed_message
    if not customer_message or not customer_message.strip():
        yield "Please record audio or enter a fallback customer message.", transcript, None
        return

    final_output = ""
    assistant_text = ""

    for partial in stream_triage(customer_message, order_id, model):
        final_output = partial
        if "### Assistant recommendation\n" in partial:
            assistant_text = partial.split("### Assistant recommendation\n", 1)[1].strip()
        yield partial, transcript, None

    if not speak_response:
        yield final_output, transcript, None
        return

    try:
        audio_response = text_to_speech_locally(assistant_text)
    except Exception as exc:
        yield final_output + f"\n\n_TTS generation failed: {exc}_", transcript, None
        return

    if not audio_response:
        yield final_output + "\n\n_TTS is unavailable on this machine._", transcript, None
        return

    yield final_output, transcript, audio_response

## Gradio UI

Run this cell to launch the prototype. Try the sample order IDs `A1001`, `A1002`, `A1003`, or `A1004`. The first tab is text-only; the second tab supports microphone or uploaded audio if local Whisper is installed.

In [ ]:
try:
    demo.close()
except NameError:
    pass
except Exception as exc:
    print(f"Previous Gradio app could not be closed cleanly: {exc}")

sample_tickets = [
    ["My earbuds arrived damaged and I need a replacement before Friday. This is really frustrating.", "A1001", DEFAULT_MODEL],
    ["Where is my charger? The tracking has not updated and I paid for express shipping.", "A1002", DEFAULT_MODEL],
    ["I want to return the standing desk converter. The box is open but I did not use it.", "A1004", DEFAULT_MODEL],
    ["Can you help me understand if the espresso kit has replacement parts?", "A1003", DEFAULT_MODEL],
]

with gr.Blocks(title="Local AI Support Triage Assistant") as demo:
    gr.Markdown("# Local AI Support Triage Assistant")
    gr.Markdown("Prototype for e-commerce support teams: classify a customer message, look up mock order context, and stream a recommended agent response from a local Ollama model.")

    with gr.Tabs():
        with gr.Tab("Text ticket"):
            with gr.Row():
                with gr.Column(scale=2):
                    customer_message = gr.Textbox(
                        label="Customer message",
                        lines=7,
                        placeholder="Paste the customer's message here...",
                    )
                    order_id = gr.Textbox(label="Order ID", placeholder="Example: A1001")
                    model = gr.Dropdown(
                        label="Local Ollama model",
                        choices=MODEL_OPTIONS,
                        value=DEFAULT_MODEL,
                    )
                    submit = gr.Button("Triage ticket", variant="primary")
                with gr.Column(scale=3):
                    output = gr.Markdown(label="Triage output")

            gr.Examples(
                examples=sample_tickets,
                inputs=[customer_message, order_id, model],
            )

            submit.click(
                fn=stream_triage,
                inputs=[customer_message, order_id, model],
                outputs=output,
            )

        with gr.Tab("Audio ticket"):
            with gr.Row():
                with gr.Column(scale=2):
                    audio_ticket = gr.Audio(
                        label="Record or upload customer audio",
                        sources=["microphone", "upload"],
                        type="filepath",
                    )
                    fallback_message = gr.Textbox(
                        label="Fallback text if no audio is provided",
                        lines=4,
                        placeholder="Optional: type the customer message here instead...",
                    )
                    audio_order_id = gr.Textbox(label="Order ID", placeholder="Example: A1001")
                    audio_model = gr.Dropdown(
                        label="Local Ollama model",
                        choices=MODEL_OPTIONS,
                        value=DEFAULT_MODEL,
                    )
                    speak_response = gr.Checkbox(label="Generate local audio response with macOS say", value=True)
                    audio_submit = gr.Button("Triage audio ticket", variant="primary")
                with gr.Column(scale=3):
                    transcript_output = gr.Textbox(label="Local transcript", lines=4)
                    audio_output = gr.Markdown(label="Triage output")
                    spoken_output = gr.Audio(label="Spoken assistant response", type="filepath")

            audio_submit.click(
                fn=stream_triage_with_audio,
                inputs=[audio_ticket, fallback_message, audio_order_id, audio_model, speak_response],
                outputs=[audio_output, transcript_output, spoken_output],
            )

demo.queue()
demo.launch(
    prevent_thread_lock=True,
    show_error=True,
    inbrowser=True,
    inline=False,
)

## How This Maps to the Week 2 Exercise

- **Gradio UI:** the final cell creates an interactive web app.
- **Streaming:** `stream_triage` yields partial responses as Ollama generates tokens.
- **System prompt expertise:** `SYSTEM_PROMPT` defines the assistant as a senior support operations helper.
- **Model switching:** the dropdown switches between local Ollama models.
- **Tool use:** `classify_ticket`, `lookup_order`, and `relevant_policies` provide structured business context before the LLM responds.
- **Optional audio:** the audio tab transcribes microphone or uploaded audio locally with Whisper if installed, then speaks the final answer using macOS `say`.

Possible next improvements: connect real Shopify/WooCommerce order data, add refund authorization rules, log agent decisions, or add audio input for spoken support tickets.